# Import semua library yang diperlukan

link google colab: https://colab.research.google.com/drive/1hEZ5lrQlNwFbysSp-I58UYCgjfX-jzvc?usp=sharing

In [1]:
import numpy as np
import pandas as pd
import scipy.stats.mstats

from google.colab import drive

# Muat dataset

In [2]:
url = 'https://drive.google.com/uc?export=download&id=1LfQWProB0VjWN5q8bKuRIgn-stULfIRo'

df = pd.read_csv(url)

# Eksplorasi awal

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


In [4]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [5]:
df.isnull().sum()

,0
id,0
luas_m2,18
harga_juta,17
kota,0
kamar,10
tahun_bangun,0
kondisi,0


# Hapus baris duplikat

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
# sebenarnya sudah terlihat di atas bahwa tidak ada baris duplikat
df.drop_duplicates(inplace=True)

# Normalisasi string

In [8]:
print('Isi kolom "kota":\n', df['kota'].unique())
print('Isi kolom "kondisi":\n', df['kondisi'].unique())

Isi kolom "kota":
 ['jogja' 'Medan' 'Depok' 'YGY' 'Jakarta' 'jakarta' 'Yogyakarta' 'Bandung'
 'Surabaya' 'dpk' 'sby' 'Makassar' 'mdn' 'medan' 'Semarang' 'semarang'
 'yogyakarta' 'Jogja' 'JAKARTA' 'Smg' 'DEPOK' 'Bdg' 'makassar' 'surabaya'
 'MAKASSAR' 'depok' 'bandung' 'Bandung ' 'SURABAYA' 'Mksr' ' Jakarta']
Isi kolom "kondisi":
 ['baik' 'Bagus' 'Sedang' 'baik sekali' 'SEDANG' 'sedang' 'BAIK' 'rusak'
 'cukup' 'Baik' 'Cukup' 'perlu renovasi' 'bagus' 'jelek' 'RUSAK']


In [9]:
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.lower()

In [11]:
print('Isi kolom "kota":\n', df['kota'].value_counts(), '\n')
print('Isi kolom "kondisi":\n', df['kondisi'].value_counts())

Isi kolom "kota":
 kota
Medan         18
Makassar      18
Bandung       16
Yogyakarta    14
Jakarta       13
Depok         10
Surabaya      10
Semarang       9
Jogja          8
Ygy            3
Sby            2
Dpk            2
Mdn            2
Smg            2
Mksr           2
Bdg            1
Name: count, dtype: int64 

Isi kolom "kondisi":
 kondisi
baik              58
sedang            37
rusak             12
bagus              7
baik sekali        7
cukup              6
jelek              2
perlu renovasi     1
Name: count, dtype: int64


In [12]:
#Ganti nama kota yang menggunakan singkatan menjadi format terstandar

mapping_kota = {
    'Mdn': 'Medan',
    'Mksr': 'Makassar',
    'Bdg': 'Bandung',
    'Ygy': 'Yogyakarta',
    'Jogja': 'Yogyakarta',
    'Sby': 'Surabaya',
    'Dpk': 'Depok',
    'Smg': 'Semarang'
}

df['kota'] = df['kota'].replace(mapping_kota)

#Kategori kondisi disederhanakan untuk mempermudah klasifikasi
mapping_kondisi = {
    'perlu renovasi': 'rusak',
    'cukup': 'sedang',
    'bagus': 'baik',
}

df['kota'] = df['kota'].replace(mapping_kota)
df['kondisi'] = df['kondisi'].replace(mapping_kondisi)

print('Isi kolom "kota" setelah diperbaiki:\n', df['kota'].value_counts(), '\n')
print('Isi kolom "kondisi" setelah diperbaiki:\n', df['kondisi'].value_counts())

Isi kolom "kota" setelah diperbaiki:
 kota
Yogyakarta    25
Medan         20
Makassar      20
Bandung       17
Jakarta       13
Depok         12
Surabaya      12
Semarang      11
Name: count, dtype: int64 

Isi kolom "kondisi" setelah diperbaiki:
 kondisi
baik           65
sedang         43
rusak          13
baik sekali     7
jelek           2
Name: count, dtype: int64


# Imputasi missing values

In [13]:
'''
data null terdapat pada kolom luas_m2, harga_juta, kamar
pada kolom luas_m2 dan harga_juta akan diisi dengan median
pada kolom kamar akan diisi dengan modus (meskipun bertipa data numerik/float namun ini termasuk data kategorikal jadi menggunakan modus)
'''

df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

In [14]:
df.isnull().sum()

,0
id,0
luas_m2,0
harga_juta,0
kota,0
kamar,0
tahun_bangun,0
kondisi,0


# Tangani outlier

In [15]:
#Tangani outlier dengan IQR Fence pada kolom harga_juta dan luas_m2 (di sini saya melihat ada anomali data pada tahun_bangun maka akan saya proses untuk menghasilkan data yang lebih bersih)

def deteksi_outlier(df, kolom):
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    outliers = df[(df[kolom] < batas_bawah) | (df[kolom] > batas_atas)]

    return batas_bawah, batas_atas, outliers

batas_bawah_harga, batas_atas_harga, outliers_harga = deteksi_outlier(df, 'harga_juta')
batas_bawah_luas, batas_atas_luas, outliers_luas = deteksi_outlier(df, 'luas_m2')
batas_bawah_tahun_bangun, batas_atas_tahun_bangun, outliers_tahun_bangun = deteksi_outlier(df, 'tahun_bangun')

print(f"Pada kolom harga_juta batas bawah Q1: {batas_bawah_harga} dan batas atas Q3: {batas_atas_harga}")
print('Outliers pada kolom "harga_juta":')
print(outliers_harga, "\n")

print(f"Pada kolom luas_m2 batas bawah Q1: {batas_bawah_luas} dan batas atas Q3: {batas_atas_luas}")
print('Outliers pada kolom "luas_m2":')
print(outliers_luas, "\n")

print(f"Pada kolom tahun_bangun batas bawah Q1: {batas_bawah_tahun_bangun} dan batas atas Q3: {batas_atas_tahun_bangun}")
print('Outliers pada kolom "tahun_bangun":')
print(outliers_tahun_bangun)

Pada kolom harga_juta batas bawah Q1: -422.75 dan batas atas Q3: 1719.25
Outliers pada kolom "harga_juta":
      id  luas_m2  harga_juta     kota  kamar  tahun_bangun      kondisi
26    27    113.8      -500.0  Bandung    6.0          1993  baik sekali
79    80    193.8      1882.0  Jakarta    2.0          1986       sedang
101  102    221.6  99999999.0    Medan    4.0          2022       sedang 

Pada kolom luas_m2 batas bawah Q1: -145.225 dan batas atas Q3: 512.9749999999999
Outliers pada kolom "luas_m2":
      id  luas_m2  harga_juta        kota  kamar  tahun_bangun kondisi
103  104   9500.0       179.0  Yogyakarta    4.0          1985    baik 

Pada kolom tahun_bangun batas bawah Q1: 1960.5 dan batas atas Q3: 2042.5
Outliers pada kolom "tahun_bangun":
      id  luas_m2  harga_juta     kota  kamar  tahun_bangun kondisi
72    73    193.8       894.0  Bandung    6.0          1890   rusak
83    84    218.5       696.0  Bandung    2.0          2099  sedang
102  103    178.5       655.0 

terlihat pada kolom "harga_juta" terdapat harga minus (-) serta harga yang sangat tinggi yang merupakan data outlier juga.
sedangkan pada kolom "luas_m2" terdapat luas yang outlier yang tinggi sekali.
ini harus diisi dengan data

cek lagi apakah ada nilai minus pada kolom "harga_juta":

In [16]:
df[df['harga_juta'] < 0]

,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
26,27,113.8,-500.0,Bandung,6.0,1993,baik sekali


karena data hasil IQR Fence masih ada minus pada kolom 'harga_juta' dan 'luas_m2' saya mengganti menjadi nilai positif, dengan asumsi salah input di sini. untuk memastikan coba lihat distribusi data menggunakan array:



In [17]:
# cek distribusi data
print(np.sort(df['harga_juta'].unique()), '\n')
print(np.sort(df['luas_m2'].unique()), '\n')
print(np.sort(df['tahun_bangun'].unique()), '\n')
print(np.sort(df['kamar'].unique()))

[-5.0000000e+02  0.0000000e+00  1.3900000e+02  1.5600000e+02
  1.7000000e+02  1.7100000e+02  1.7600000e+02  1.7800000e+02
  1.7900000e+02  1.8100000e+02  1.8300000e+02  1.9500000e+02
  2.0200000e+02  2.0600000e+02  2.0800000e+02  2.1000000e+02
  2.3100000e+02  2.3700000e+02  2.4700000e+02  2.5200000e+02
  2.5800000e+02  3.0400000e+02  3.0700000e+02  3.1500000e+02
  3.3300000e+02  3.4500000e+02  3.4900000e+02  3.8000000e+02
  3.8200000e+02  4.2200000e+02  4.2400000e+02  4.5500000e+02
  4.5600000e+02  4.5700000e+02  4.9500000e+02  5.0300000e+02
  5.0600000e+02  5.1100000e+02  5.2200000e+02  5.2900000e+02
  5.3200000e+02  5.7500000e+02  5.9500000e+02  6.3000000e+02
  6.3100000e+02  6.3400000e+02  6.4100000e+02  6.5500000e+02
  6.6400000e+02  6.6500000e+02  6.7700000e+02  6.9100000e+02
  6.9600000e+02  7.3500000e+02  7.4600000e+02  7.5100000e+02
  7.6100000e+02  7.6700000e+02  7.8300000e+02  8.0200000e+02
  8.1400000e+02  8.1800000e+02  8.1900000e+02  8.6300000e+02
  8.8000000e+02  8.94000

dari distribusi data di atas jika nilai negatif diganti positif maka masuk akal sehingga semua data negatif saya ganti menjadi positif karena

In [18]:
df.loc[df['harga_juta'] < 0, 'harga_juta'] = -df['harga_juta']

begitu juga dengan nilai minus pada kolom 'luas_m2' diganti menjadi positif

In [19]:
df.loc[df['luas_m2'] < 0, 'luas_m2'] = -df['luas_m2']

pada tahun_bangun terdapat tahun 2099 dan 9999, maka akan dinormalisasi dengan menjadikan nilai modus

In [20]:
df.loc[df['tahun_bangun'] > 2026, 'tahun_bangun'] = df['tahun_bangun'].mode()[0]

In [22]:
df['tahun_bangun'].value_counts()

,count
tahun_bangun,
1993,8
2002,6
2011,5
2016,5
2013,5
1986,5
2017,4
1998,4
2005,4


In [23]:
df.describe()

,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,130.000000,1.300000e+02,130.000000,130.000000
mean,65.500000,258.174615,7.699124e+05,3.246154,2000.238462
std,37.671829,821.708763,8.770520e+06,1.826003,15.466294
min,1.000000,40.400000,0.000000e+00,1.000000,1890.000000
25%,33.250000,101.600000,3.920000e+02,1.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,3.000000,2002.000000
75%,97.750000,266.150000,9.160000e+02,5.000000,2011.000000
max,130.000000,9500.000000,1.000000e+08,6.000000,2023.000000


## Mengganti data outlier dengan IQR Fence:

In [24]:
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

In [25]:
# periksa kembali

batas_bawah_harga_new, batas_atas_harga_new, outliers_harga_new = deteksi_outlier(df, 'harga_juta')
batas_bawah_luas_new, batas_atas_luas_new, outliers_luas_new = deteksi_outlier(df, 'luas_m2')

print(f"Jumlah outlier di 'harga_juta' setelah clipping: {len(outliers_harga_new)}")
print(f"Jumlah outlier di 'luas_m2' setelah clipping: {len(outliers_luas_new)}")
print(f"Pada kolom harga_juta batas bawah Q1: {batas_bawah_harga} dan batas atas Q3: {batas_atas_harga}")
print('Outliers pada kolom "harga_juta":')
print(outliers_harga, "\n")

print(f"Pada kolom luas_m2 batas bawah Q1: {batas_bawah_luas} dan batas atas Q3: {batas_atas_luas}")
print('Outliers pada kolom "luas_m2":')
print(outliers_luas, "\n")

print(f"Pada kolom tahun_bangun batas bawah Q1: {batas_bawah_tahun_bangun} dan batas atas Q3: {batas_atas_tahun_bangun}")
print('Outliers pada kolom "tahun_bangun":')
print(outliers_tahun_bangun)

Jumlah outlier di 'harga_juta' setelah clipping: 0
Jumlah outlier di 'luas_m2' setelah clipping: 0
Pada kolom harga_juta batas bawah Q1: -422.75 dan batas atas Q3: 1719.25
Outliers pada kolom "harga_juta":
      id  luas_m2  harga_juta     kota  kamar  tahun_bangun      kondisi
26    27    113.8      -500.0  Bandung    6.0          1993  baik sekali
79    80    193.8      1882.0  Jakarta    2.0          1986       sedang
101  102    221.6  99999999.0    Medan    4.0          2022       sedang 

Pada kolom luas_m2 batas bawah Q1: -145.225 dan batas atas Q3: 512.9749999999999
Outliers pada kolom "luas_m2":
      id  luas_m2  harga_juta        kota  kamar  tahun_bangun kondisi
103  104   9500.0       179.0  Yogyakarta    4.0          1985    baik 

Pada kolom tahun_bangun batas bawah Q1: 1960.5 dan batas atas Q3: 2042.5
Outliers pada kolom "tahun_bangun":
      id  luas_m2  harga_juta     kota  kamar  tahun_bangun kondisi
72    73    193.8       894.0  Bandung    6.0          1890   rusak

### Periksa kembali outlier setelah clipping

In [26]:
batas_bawah_harga_juta_new, batas_atas_harga_juta_new, outliers_harga_juta_new = deteksi_outlier(df, 'harga_juta')
batas_bawah_luas_m2_new, batas_atas_luas_m2_new, outliers_luas_m2_new = deteksi_outlier(df, 'luas_m2')
batas_bawah_tahun_bangun_new, batas_atas_tahun_bangun_new, outliers_tahun_bangun_new = deteksi_outlier(df, 'tahun_bangun')

print(f"Jumlah outlier di 'harga_juta' setelah clipping: {len(outliers_harga_juta_new)}")
print(f"Jumlah outlier di 'luas_m2' setelah clipping: {len(outliers_luas_m2_new)}")
print(f"Jumlah outlier di 'tahun_bangun' setelah clipping: {len(outliers_tahun_bangun_new)}")

# Menampilkan perubahan data dari outliers yang sudah di-clipping
print('\nDataframe setelah clipping (baris-baris yang sebelumnya outlier di harga_juta):')
display(df.loc[outliers_harga.index])

print('\nDataframe setelah clipping (baris-baris yang sebelumnya outlier di luas_m2):')
display(df.loc[outliers_luas.index])

print('\nDataframe setelah clipping (baris-baris yang sebelumnya outlier di tahun_bangun):')
display(df.loc[outliers_tahun_bangun.index])

Jumlah outlier di 'harga_juta' setelah clipping: 0
Jumlah outlier di 'luas_m2' setelah clipping: 0
Jumlah outlier di 'tahun_bangun' setelah clipping: 0

Dataframe setelah clipping (baris-baris yang sebelumnya outlier di harga_juta):


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
26,27,113.8,500.0,Bandung,6.0,1993.0,baik sekali
79,80,193.8,1702.0,Jakarta,2.0,1986.0,sedang
101,102,221.6,1702.0,Medan,4.0,2022.0,sedang



Dataframe setelah clipping (baris-baris yang sebelumnya outlier di luas_m2):


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
103,104,512.975,179.0,Yogyakarta,4.0,1985.0,baik



Dataframe setelah clipping (baris-baris yang sebelumnya outlier di tahun_bangun):


,id,luas_m2,harga_juta,kota,kamar,tahun_bangun,kondisi
72,73,193.8,894.0,Bandung,6.0,1961.625,rusak
83,84,218.5,696.0,Bandung,2.0,1993.000,sedang
102,103,178.5,655.0,Medan,3.0,1993.000,sedang


# Validasi akhir

In [27]:
print(f"Missing value pada data: \n{df.isnull().sum()} \n")
print(f"Duplikat data pada data: {df.duplicated().sum()}")

Missing value pada data: 
id              0
luas_m2         0
harga_juta      0
kota            0
kamar           0
tahun_bangun    0
kondisi         0
dtype: int64 

Duplikat data pada data: 0


In [28]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

In [29]:
print('Shape akhir:', df.shape)

Shape akhir: (130, 7)


# Ekspor data ke dalam format CSV

In [30]:
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Dataset bersih tersimpan!


# Akses API JSONPlaceholder dan simpan respons sebagai DataFrame

In [31]:
import requests
from pandas import json_normalize

# JSONPlaceholder
URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)

# Cek status code
if response.status_code == 200:
  data = response.json()
  df = json_normalize(data, sep='_')
  print(df[['id','name','email','address_city']])
else:
  print(f'Error: {response.status_code}')

# API dengan parameter (query string)
params = {'userId': 1}   # filter by user
posts = requests.get("https://jsonplaceholder.typicode.com/posts",
params=params).json()
df_posts = pd.DataFrame(posts)

   id                      name                      email    address_city
0   1             Leanne Graham          Sincere@april.biz     Gwenborough
1   2              Ervin Howell          Shanna@melissa.tv     Wisokyburgh
2   3          Clementine Bauch         Nathan@yesenia.net   McKenziehaven
3   4          Patricia Lebsack  Julianne.OConner@kory.org     South Elvis
4   5          Chelsey Dietrich   Lucio_Hettinger@annie.ca      Roscoeview
5   6      Mrs. Dennis Schulist    Karley_Dach@jasper.info   South Christy
6   7           Kurtis Weissnat     Telly.Hoeger@billy.biz       Howemouth
7   8  Nicholas Runolfsdottir V       Sherwood@rosamond.me       Aliyaview
8   9           Glenna Reichert    Chaim_McDermott@dana.io  Bartholomebury
9  10        Clementina DuBuque     Rey.Padberg@karina.biz     Lebsackbury
